# Find Waldo — Dataset Exploration
This notebook visualises the raw data and training results.

**Credits:**
- Hey-Waldo dataset: https://github.com/vc1492a/Hey-Waldo
- HereIsWally: https://github.com/tadejmagajna/HereIsWally
- YOLOv8: https://github.com/ultralytics/ultralytics

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np

ROOT = Path('..').resolve()
RAW  = ROOT / 'data' / 'raw'
PROC = ROOT / 'data' / 'processed'

## 1. Browse Hey-Waldo 64×64 patches

In [ ]:
waldo_dir = RAW / 'Hey-Waldo' / '64' / 'Waldo'
not_dir   = RAW / 'Hey-Waldo' / '64' / 'NotWaldo'

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle('Top row: Waldo  |  Bottom row: Not Waldo', fontsize=14)

for i, p in enumerate(sorted(waldo_dir.glob('*.jpg'))[:8]):
    axes[0, i].imshow(Image.open(p))
    axes[0, i].axis('off')

for i, p in enumerate(sorted(not_dir.glob('*.jpg'))[:8]):
    axes[1, i].imshow(Image.open(p))
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## 2. Browse HereIsWally full scenes with bounding boxes

In [ ]:
import xml.etree.ElementTree as ET

scenes = sorted((RAW / 'HereIsWally').rglob('*.jpg'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, scene in zip(axes.flat, scenes):
    img = np.array(Image.open(scene))
    ax.imshow(img)
    ax.set_title(scene.stem, fontsize=9)
    ax.axis('off')

    xml = scene.with_suffix('.xml')
    if xml.exists():
        tree = ET.parse(xml)
        for obj in tree.getroot().iter('object'):
            bb = obj.find('bndbox')
            x1 = float(bb.find('xmin').text)
            y1 = float(bb.find('ymin').text)
            x2 = float(bb.find('xmax').text)
            y2 = float(bb.find('ymax').text)
            rect = patches.Rectangle((x1,y1), x2-x1, y2-y1,
                                      linewidth=3, edgecolor='red', facecolor='none')
            ax.add_patch(rect)

plt.tight_layout()
plt.show()

## 3. Training curves (after training)

In [ ]:
import pandas as pd

results_csv = ROOT / 'models' / 'waldo_yolov8n' / 'results.csv'
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    df.plot('epoch', 'metrics/mAP50(B)', ax=axes[0], title='mAP@0.5', legend=False)
    df.plot('epoch', 'metrics/precision(B)', ax=axes[1], title='Precision', legend=False)
    df.plot('epoch', 'metrics/recall(B)', ax=axes[2], title='Recall', legend=False)
    for ax in axes:
        ax.set_xlabel('Epoch')
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Train the model first: python src/train.py')

## 4. Run prediction on a custom image

In [ ]:
from ultralytics import YOLO
import requests
from io import BytesIO

BEST_PT = ROOT / 'models' / 'waldo_yolov8n' / 'weights' / 'best.pt'

# Replace with your own image path or URL
TEST_IMAGE_PATH = None  # e.g. Path('/path/to/scene.jpg')

if BEST_PT.exists() and TEST_IMAGE_PATH is not None:
    model = YOLO(str(BEST_PT))
    results = model.predict(source=str(TEST_IMAGE_PATH), conf=0.25, iou=0.45)
    results[0].show()  # opens window or displays inline in Jupyter
else:
    print('Set TEST_IMAGE_PATH and make sure the model is trained.')